In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits
import os
import json
import odos.optimal_period as op

## Introduction
In this turtorial we will see how can we utilize the code to predict possible future transits for an exoplanet with a non resolved period. We will use TOI_2076c as en example. 
#### Prerequisites
- Fitted midtimes of the observations, for simplicity
- The maximum possible period, and an epoch reference, with their errors priorly loaded on a json file, as in `2076c_params.json`
- The transit duration in days, as in `2076c_params.json`
- Parameters regarding the observing campain, like observatory locations, and observation periods. 

Initially we load the `.json` file with the mentioned parameters, and load it. 

In [ ]:
pl_nam = 'TOI-2076.02'
with open('2076c_params.json', 'r') as f:
    pp = json.load(f)


Then, we create the UnknownPeriodPlanet object, providing the transit duration, depth, period, epoch reference (mentioned as mid_time for historical reasons.) and the observation midtimes. 

! Hint: for only 2 midtimes, P_max and reference_midtime are not necessary, otherwise they are used in the linear fit of the midtimes. So, in theory they could be set to None, only for 2 midtimes. In any case they will be calculated when calling the `possible_periods` function. 

In [ ]:
my_planet = op.UnknownPeriodPlanet(pl_nam,
                                   P_min=15, # limit that the code will stop to search for smaller and smaller periods 
                                   transit_duration=pp['transit_duration'], # for plotting reasons possible half transits
                                   transit_depth=pp['transit_depth']/1000000 # for plotting reasons
                )

my_planet.transit_edge_limit = my_planet.transit_duration/2 # for possible half-transits that have been missed
my_planet.P_max = [pp['period'],pp['period_err']]
my_planet.reference_midtime = [pp['mid_time'],pp['mid_time_err']]
# my_planet.P_max = None
# my_planet.reference_midtime = None


my_planet.t_midtimes = [[2458748.69458, 0.001], [2458937.8222, 0.1]] # the observed midtimes


We are next going to add observations data, that require the time, flux, and flux-error. This is essential, as the code uses the observations span (taking into account gaps) in order to calculate possible periods aliases, as well as probable missed transits (meaning that for a period alias, we have data observations for a predicted midtime, but it is not added to the `my_planet.t_midtimes`).

Below we provide the TESS PDCSAP flux. Of course the transits are not detrended, and can potentially look a bit ugly, but we can at least visually confirm they are there.  

In [ ]:
my_planet.add_observation?

In [ ]:

pl_path = 'mast_data/'

pathlist = os.listdir(pl_path)
pathlist.sort()
for file_path in pathlist:
    print(file_path)
    tess_dat = fits.open(os.path.join(pl_path, file_path))
    df231 = pd.DataFrame(tess_dat[1].data.T)
    df23 = df231[['TIME', 'PDCSAP_FLUX','PDCSAP_FLUX_ERR']]
    df23 = df23.dropna()
    df23 = df23.reset_index(drop=True)
    tim = np.array(df23['TIME'])
    flux = np.array(df23['PDCSAP_FLUX'])
    flux_err = np.array(df23['PDCSAP_FLUX_ERR'])
    my_planet.add_observation(tim + 2457000, flux, flux_err) # we can also add binning, used to identify gaps in the data, and plot_label

We can now plot the possible transits that are found in the data

In [ ]:
my_planet.observations[0]['plot_label'] = 'TESS Sect 16'
my_planet.observations[1]['plot_label'] = 'TESS Sect 23'
with plt.rc_context({'axes.labelsize': 12 , 'axes.linewidth': 2, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 10, 'axes.titlesize': 20}):
    my_planet.phase_plot()

Grat! we can see them!

We are ready to calculate the possible periods providing these midtimes. 

In [ ]:
my_planet.possible_periods()
my_planet.all_possible_periods_df

Now we have calculated all the possible period aliases, from the minimum period that we defined, until the maximum one. 

We can notice that some period aliases, have at least one `missed_transits_total` - meaning it would be a bit safer to check if at this possible transit, we could actually have something

We do this with the `sanity_plot` function, for the 12th period order that has 1 `missed_transit`. We also plot the 9, 10, 11 ordres. 

In [ ]:
my_planet.sanity_plot(
                    plot_periods_orders=[9, 10, 11, 12],
                    plot_observation=1
                )

We see that alias 2 is provavly not a missed transit. We can manually exclude the periods from the all_possible_periods_df, and pass the new dataframe to the object, only with the periods we want to use: 

In [ ]:
periods2use = my_planet.all_possible_periods_df.query('missed_transits_total== 0')
my_planet.setup_possible_periods(periods2use)

In [ ]:
my_planet.possible_periods_df

Then we calculate the observation bias (for the number of transits that we have - it is interesting to check how this changes depending on the dataset and the number of transits that we have) as well as any other prior probability we might know for this planet. 

Then we calculate the intermediate period alias probability step, before we can to each individual transit probability period.

In [ ]:
my_planet.obs_bias_period_prob(N_transits=2)
prior_prob = np.ones(len(my_planet.possible_periods_df))
my_planet.prior_prob_period = prior_prob


my_planet.best_assumed_period()


In [ ]:
my_planet.possible_periods_df

The above analysis, creates the following:
- `obs_bias_period_prob` -> The observation bias, given our data, and midtimes
- `final_prob_period` -> Normalized probability for each alias.
- `N_is_transit` -> How many period aliases are we left with, after a positive transit observation of this period alias?
- `N_no_transit` -> How many period aliases are we left with, after a negative transit observation of this period alias?
- `prob_is_transit` -> What is the probability an observation with this alias is a transit?


# Estimating the observations
next step, is to calculate all the possible transits within our observation period (s). More than one time spans can be added 


In [ ]:
## CALCULATE TRANSIT FOR START AND END DATE IN BJD ##
OBSERV_START= '2025-10-01'
OBSERV_END = '2026-06-01'
my_planet.all_transits_multiple_periods([[OBSERV_START,OBSERV_END]],tw = 1/24)


Of course, all this will not be observable from any point in Earth, so we have to add the Star Positions, and some observatory locations as seen below. The observable transits is done using `astroplan` package.

In [ ]:
STAR_RA = pp['RA']
STAR_DEC = pp['DEC']

## OBSERVATORY PARAMETERS ##
###TEIDE OBSERVATORY ###
OBS_LAT = 28.3011
OBS_LONG = -16.5103
OBS_ALT = 2400

# MAUNA KEA OBSERVATORY ###
OBS_LAT1 = 19.8242
OBS_LONG1 = -155.477
OBS_ALT1 = 4205

In [ ]:
my_planet.initialize_star(STAR_RA,STAR_DEC)
my_planet.initalize_observatory_locations([
                [OBS_LAT,OBS_LONG,OBS_ALT, 'TEIDE'], 
                [OBS_LAT1,OBS_LONG1,OBS_ALT1, 'MAUNA KEA']
                ]
                )

And we are ready to calculate all the observable transits from an observatory. 

In [ ]:
my_planet.observable_transits_multiple_periods(min_h_horizon=10, min_moon_separation = 0)

In [ ]:
my_planet.all_transits_df.head()

of course, trying to choose among the many transits present in a long observing campaign can be messy, and we can automatically try to find the best transits to observe as much period aliases as possible. 

First, we can query the observable transits datagrame, for all observations that can take place from TEIDE, if we have a full transit, or if we can only observe the `ingress` or `egress` as shown below: 

In [ ]:
observable_transits = my_planet.all_transits_df[[x in ['full', 'ingress', 'egress'] for x in my_planet.all_transits_df["TEIDE"]]]
observable_transits.reset_index(drop = True, inplace = True)

In [ ]:
pers, trdf = my_planet.max_P_coverage(
    N_observations = 5,  # how many observations can we allocate
    df_transits = observable_transits, # leave to None if all the transtts can be calculates
    max_N_periods=35 # internal to the code. 
)

trdf.sort_values(by = 'start_obs', inplace=True)
trdf.reset_index(drop = True, inplace=True)
trdf

# The period aliases - temperature plot
Finally, in order to better understand what all these period aliases can mean for the planetary properties, we can make an extra plot showing the period probability as a function of the period alias, color-coded with the temprature. 

In [ ]:
## find black body equilibrium temperature
def semi_major_axis(P, M_s):
    return ((6.67*10**(-11)*(M_s*1.919*10**30)*(P*24*60*60)**2)/4/(np.pi)**2)**(1/3)/(6.96*10**8)

radius_sun = 0.94
mass_sun = 1.015
temp_star = 5798

temps = []
for per in my_planet.possible_periods_values:
    a = semi_major_axis(per,mass_sun)
    temp = temp_star*np.sqrt(radius_sun/(2*a))
    temps.append(temp)
colors_label = ['T>800', '400<T<800', '270<T<400', 'T<270']
colors_list = ['#FB4B37', '#A47F3D','#3ADCC6', '#148BA7']
colors_periods = []

xi = np.array(list(my_planet.possible_periods_df['period']))
yi = np.array(list(100*my_planet.possible_periods_df['final_prob_period']))

xil = [[], [], [], []]
yil = [[], [], [], []]

for i,temp in enumerate(temps):
    if temp < 270:
        xil[3].append(xi[i])
        yil[3].append(yi[i])
    elif temp < 400:
        xil[2].append(xi[i])
        yil[2].append(yi[i])
    elif temp < 800:
        xil[1].append(xi[i])
        yil[1].append(yi[i])
    else:
        xil[0].append(xi[i])
        yil[0].append(yi[i])

In [ ]:
from matplotlib.ticker import ScalarFormatter


# create a dark grey figure
fig, ax = plt.subplots(figsize=(10,5), constrained_layout=True)

for i, x in enumerate(xil):
    if len(x) > 0:
        bars = ax.bar(
            x = xil[i],
            height=yil[i],
            color = colors_list[i],
            alpha = 0.8, zorder = 400,
            width = np.array(xil[i])/50, 
            label = colors_label[i]
            )
#

ax.set_xscale('log')

ax.xaxis.set_major_formatter(ScalarFormatter())
ax.minorticks_off()


ax.set_xticks([189.13,  94.57, 63.04, 47.28, 37.85, 31.52,  27.09,  23.64, 21.01,  18.91,  17.19])

ax.set(ylim=(0,1.1*max(yi)))
plt.legend(fontsize="12")
ax.set_title('Probability of Periods')
ax.set_xlabel('Period [days]')
ax.set_ylabel('% Probability')



